# Software Release Readiness Team

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 60 minutes

A release gate should be able to explain both “no-go” and “go.” This showcase
uses a real temporary Python release candidate, real subprocess test execution,
a static security rule, a bounded patch, and locally stored Praval spans.


## Problem and outcome

Ledger Candidate 1.4.0 has a percentage-calculation defect, an unsafe expression
evaluator, and incomplete security release notes. The team must discover those
facts independently, refuse the initial release, apply one allow-listed repair
inside a temporary copy, rerun the checks, and publish a release dossier.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "examples" / "notebooks"]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, setup_case_study, show_artifact, show_callout,
    show_flow, show_json, show_message_graph, show_roles, show_scorecard,
    show_spore, show_timeline, stage,
)

setup_case_study("Software Release Readiness Team")


## Course prerequisites

Complete the Reef pipeline, parallel fan-in, tools, and production observability
lessons. The notebook runs commands as argument lists with `shell=False`; the
fixture is never edited in place.


## Architecture and responsibilities

```text
release_candidate → review_assignment × 4 + change review → review_finding
                  → initial_gate_decision → remediation_plan
                  → bounded_patch_request → verification_result
                  → final_gate_decision → release_dossier
```

Five review findings feed the gate. Four are assigned; the change analyst starts
directly from the candidate so the requested fan-out remains four.


In [ ]:
show_flow(
    ("Candidate", "temporary copy", "spore"),
    ("Reviewers", "tests, security, package, docs", "agent"),
    ("Gatekeeper", "initial no-go", "human"),
    ("Safe patch", "allow-listed repair", "tool"),
    ("Verifier", "rerun every check", "agent"),
    ("Dossier", "final go with evidence", "memory"),
)
show_roles([
    ("Release coordinator", "Assign reviews and own the dossier", "agent"),
    ("Change analyst", "Inventory candidate scope", "agent"),
    ("Test analyst", "Execute the candidate test file", "agent"),
    ("Security reviewer", "Detect unsafe dynamic evaluation", "agent"),
    ("Packaging reviewer", "Validate metadata", "agent"),
    ("Documentation reviewer", "Check security release notes", "agent"),
    ("Remediation planner", "Select the bounded patch", "agent"),
    ("Verification agent", "Apply and rerun checks", "agent"),
    ("Release gatekeeper", "Issue initial and final decisions", "agent"),
])


## Design decisions

- The committed fixture is immutable. Every run begins with `copytree` into a
  `TemporaryDirectory`.
- Tools accept only the temporary workspace and a fixed patch ID.
- Tests use the running Python interpreter and a list of arguments.
- Every tool operation creates a local span. Failed checks are recorded as
  `ERROR`; successful verification creates fresh `OK` spans.


## Implementation

First create the disposable workspace and configure observability to write into
that same directory. Resetting configuration before importing the tracer makes
the storage path explicit and inspectable.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib

from praval import agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

reset_reef()
reset_tool_registry()
fixture = find_example_asset("notebooks/fixtures/release_candidate")
workspace_context = tempfile.TemporaryDirectory(prefix="praval-release-capstone-")
workspace = Path(workspace_context.name) / "candidate"
shutil.copytree(fixture, workspace)
assert workspace.resolve() != fixture.resolve()

os.environ["PRAVAL_OBSERVABILITY"] = "on"
os.environ["PRAVAL_TRACES_PATH"] = str(Path(workspace_context.name) / "traces.sqlite3")
from praval.observability.config import reset_config
from praval.observability.storage.sqlite_store import get_trace_store, reset_trace_store
from praval.observability.tracing.tracer import get_tracer, reset_tracer

reset_config(); reset_trace_store(); reset_tracer()
tracer = get_tracer("release-readiness")
trace_store = get_trace_store()


In [ ]:
tool_events = []


def tool_span(name, **attributes):
    return tracer.start_as_current_span(f"release.tool.{name}", attributes=attributes)


@tool("release_inspect_candidate", description="Inventory a bounded release candidate.",
      category="release", shared=True)
def inspect_candidate(workspace_path: str) -> dict:
    root = Path(workspace_path)
    with tool_span("inspect_candidate", workspace=root.name) as span:
        files = sorted(path.relative_to(root).as_posix() for path in root.rglob("*") if path.is_file())
        result = {"files": files, "changelog": (root / "CHANGELOG.md").read_text()}
        span.set_status("ok"); tool_events.append({"tool": "inspect_candidate", "files": len(files)})
        return result


@tool("release_run_tests", description="Run the candidate test file without a shell.",
      category="release", shared=True)
def run_candidate_tests(workspace_path: str) -> dict:
    root = Path(workspace_path)
    with tool_span("run_tests", workspace=root.name) as span:
        result = subprocess.run(
            [sys.executable, "tests/test_ledger.py"], cwd=root, shell=False,
            text=True, capture_output=True, timeout=20, check=False,
        )
        output = (result.stdout + result.stderr)[-1200:]
        status = {"passed": result.returncode == 0, "returncode": result.returncode,
                  "output": output}
        span.set_status("ok" if status["passed"] else "error", "candidate test result")
        tool_events.append({"tool": "run_tests", "passed": status["passed"]})
        return status


@tool("release_security_scan", description="Run the allow-listed static security rules.",
      category="release", shared=True)
def security_scan(workspace_path: str) -> dict:
    source = (Path(workspace_path) / "src/ledger.py").read_text()
    with tool_span("security_scan") as span:
        findings = ["SEC-001 unsafe eval in evaluate_adjustment"] if "eval(" in source else []
        span.set_status("error" if findings else "ok", "static security result")
        result = {"passed": not findings, "findings": findings}
        tool_events.append({"tool": "security_scan", "passed": result["passed"]})
        return result


In [ ]:
@tool("release_validate_metadata", description="Validate package metadata and release notes.",
      category="release", shared=True)
def validate_metadata(workspace_path: str) -> dict:
    root = Path(workspace_path)
    with tool_span("validate_metadata") as span:
        metadata = tomllib.loads((root / "pyproject.toml").read_text())
        changelog = (root / "CHANGELOG.md").read_text()
        package_ok = metadata["project"]["name"] == "ledger-candidate" and metadata["project"]["version"] == "1.4.0"
        security_documented = "## Security" in changelog
        result = {"package_ok": package_ok, "security_documented": security_documented}
        span.set_status("ok" if all(result.values()) else "error", "metadata result")
        tool_events.append({"tool": "validate_metadata", **result})
        return result


SAFE_LEDGER = """# Ledger calculations for the fictional release candidate.
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}


def service_credit(monthly_fee: float, percent: float) -> float:
    return round(monthly_fee * percent / 100, 2)


def evaluate_adjustment(expression: str) -> float:
    def calculate(node):
        if isinstance(node, ast.Expression):
            return calculate(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](calculate(node.left), calculate(node.right))
        raise ValueError("unsupported adjustment expression")
    return float(calculate(ast.parse(expression, mode="eval")))
"""


@tool("release_apply_safe_patch", description="Apply the predefined RC-140-safe patch.",
      category="release", shared=True)
def apply_safe_patch(workspace_path: str, patch_id: str) -> dict:
    if patch_id != "RC-140-safe":
        raise ValueError("patch is not allow-listed")
    root = Path(workspace_path)
    with tool_span("apply_safe_patch", patch_id=patch_id) as span:
        (root / "src/ledger.py").write_text(SAFE_LEDGER, encoding="utf-8")
        changelog = (root / "CHANGELOG.md").read_text()
        changelog += "\n## Security\n\n- Replace dynamic evaluation with an arithmetic AST allow-list.\n"
        (root / "CHANGELOG.md").write_text(changelog, encoding="utf-8")
        span.set_status("ok"); tool_events.append({"tool": "apply_safe_patch", "patch_id": patch_id})
        return {"applied": True, "patch_id": patch_id,
                "changed": ["src/ledger.py", "CHANGELOG.md"]}


In [ ]:
REQUIRED = {"type", "domain_id", "correlation_id", "producer", "stage", "status", "payload"}


def release_message(message_type, producer, stage_name, status, payload):
    message = {
        "type": message_type, "domain_id": "ledger-candidate-1.4.0",
        "correlation_id": "release-rc-140", "producer": producer,
        "stage": stage_name, "status": status, "payload": payload,
    }
    assert set(message) == REQUIRED
    return message


message_trail, received_spores = [], []
review_findings, gate_decisions, dossiers = {}, [], []


def remember_spore(spore):
    received_spores.append(spore)
    message_trail.append(dict(spore.knowledge))


@agent("release-coordinator", responds_to=["release_candidate", "remediation_plan", "final_gate_decision"],
       auto_broadcast=False)
def release_coordinator(spore):
    remember_spore(spore)
    if spore.knowledge["type"] == "release_candidate":
        for role in ("test", "security", "packaging", "documentation"):
            broadcast(release_message("review_assignment", "release-coordinator", "assignment",
                                      "open", {"role": role, "workspace": str(workspace)}))
    elif spore.knowledge["type"] == "remediation_plan":
        broadcast(release_message("bounded_patch_request", "release-coordinator", "remediation",
                                  "approved", spore.knowledge["payload"]))
    else:
        dossier = {
            "candidate": "ledger-candidate 1.4.0", "initial_findings": review_findings,
            "remediation": "RC-140-safe", "before_after": gate_decisions,
            "residual_risks": ["The arithmetic allow-list remains intentionally small."],
            "final_gate": spore.knowledge["payload"],
        }
        dossiers.append(dossier)
        broadcast(release_message("release_dossier", "release-coordinator", "dossier",
                                  "complete", dossier))


@agent("change-analyst", responds_to=["release_candidate"],
       tools=["release_inspect_candidate"], auto_broadcast=False)
def change_analyst(spore):
    remember_spore(spore)
    result = inspect_candidate.execute_as_tool(str(workspace))
    broadcast(release_message("review_finding", "change-analyst", "change_review", "pass",
                              {"area": "change", "evidence": result["files"]}))


In [ ]:
def assignment_for(spore, role):
    return spore.knowledge["payload"]["role"] == role


@agent("test-analyst", responds_to=["review_assignment"],
       tools=["release_run_tests"], auto_broadcast=False)
def test_analyst(spore):
    if not assignment_for(spore, "test"):
        return
    remember_spore(spore)
    result = run_candidate_tests.execute_as_tool(str(workspace))
    broadcast(release_message("review_finding", "test-analyst", "test_review",
                              "pass" if result["passed"] else "fail",
                              {"area": "test", "evidence": result}))


@agent("security-reviewer", responds_to=["review_assignment"],
       tools=["release_security_scan"], auto_broadcast=False)
def security_reviewer(spore):
    if not assignment_for(spore, "security"):
        return
    remember_spore(spore)
    result = security_scan.execute_as_tool(str(workspace))
    broadcast(release_message("review_finding", "security-reviewer", "security_review",
                              "pass" if result["passed"] else "fail",
                              {"area": "security", "evidence": result}))


@agent("packaging-reviewer", responds_to=["review_assignment"],
       tools=["release_validate_metadata"], auto_broadcast=False)
def packaging_reviewer(spore):
    if not assignment_for(spore, "packaging"):
        return
    remember_spore(spore)
    result = validate_metadata.execute_as_tool(str(workspace))
    broadcast(release_message("review_finding", "packaging-reviewer", "package_review",
                              "pass" if result["package_ok"] else "fail",
                              {"area": "packaging", "evidence": result}))


@agent("documentation-reviewer", responds_to=["review_assignment"], auto_broadcast=False)
def documentation_reviewer(spore):
    if not assignment_for(spore, "documentation"):
        return
    remember_spore(spore)
    result = validate_metadata.execute_as_tool(str(workspace))
    status = "pass" if result["security_documented"] else "fail"
    broadcast(release_message("review_finding", "documentation-reviewer", "docs_review", status,
                              {"area": "documentation", "evidence": result}))


In [ ]:
@agent("release-gatekeeper", responds_to=["review_finding", "verification_result"],
       auto_broadcast=False)
def release_gatekeeper(spore):
    remember_spore(spore)
    if spore.knowledge["type"] == "review_finding":
        review_findings[spore.knowledge["producer"]] = spore.knowledge["payload"]
        if len(review_findings) != 5:
            return
        def failed_review(item):
            area, evidence = item["area"], item["evidence"]
            if area == "change":
                return False
            if area in {"test", "security"}:
                return not evidence["passed"]
            if area == "packaging":
                return not evidence["package_ok"]
            return not evidence["security_documented"]
        failed = sorted(name for name, item in review_findings.items() if failed_review(item))
        decision = {"decision": "NO-GO", "failed_reviews": failed}
        gate_decisions.append(decision)
        broadcast(release_message("initial_gate_decision", "release-gatekeeper", "initial_gate",
                                  "blocked", decision))
    else:
        verification = spore.knowledge["payload"]
        decision = {"decision": "GO" if verification["all_passed"] else "NO-GO",
                    "verification": verification}
        gate_decisions.append(decision)
        broadcast(release_message("final_gate_decision", "release-gatekeeper", "final_gate",
                                  "approved" if decision["decision"] == "GO" else "blocked", decision))


@agent("remediation-planner", responds_to=["initial_gate_decision"], auto_broadcast=False)
def remediation_planner(spore):
    remember_spore(spore)
    assert spore.knowledge["payload"]["decision"] == "NO-GO"
    plan = {"patch_id": "RC-140-safe", "workspace": str(workspace),
            "scope": ["src/ledger.py", "CHANGELOG.md"], "maximum_patches": 1}
    broadcast(release_message("remediation_plan", "remediation-planner", "plan", "complete", plan))


@agent("verification-agent", responds_to=["bounded_patch_request"],
       tools=["release_apply_safe_patch", "release_run_tests", "release_security_scan",
              "release_validate_metadata"], auto_broadcast=False)
def verification_agent(spore):
    remember_spore(spore)
    request = spore.knowledge["payload"]
    patch = apply_safe_patch.execute_as_tool(str(workspace), request["patch_id"])
    tests = run_candidate_tests.execute_as_tool(str(workspace))
    security = security_scan.execute_as_tool(str(workspace))
    metadata = validate_metadata.execute_as_tool(str(workspace))
    result = {"patch": patch, "tests": tests, "security": security, "metadata": metadata,
              "all_passed": tests["passed"] and security["passed"] and all(metadata.values())}
    broadcast(release_message("verification_result", "verification-agent", "verification",
                              "pass" if result["all_passed"] else "fail", result))


## End-to-end run

The release candidate is copied before this broadcast. Reviewer tools therefore
observe a real filesystem while the repository fixture remains unchanged. The
initial failures drive exactly one remediation cycle.


In [ ]:
agents = (
    release_coordinator, change_analyst, test_analyst, security_reviewer,
    packaging_reviewer, documentation_reviewer, remediation_planner,
    verification_agent, release_gatekeeper,
)
start_agents(
    *agents,
    initial_data=release_message(
        "release_candidate", "release-coordinator", "intake", "open",
        {"workspace": str(workspace), "version": "1.4.0"},
    ),
    channel="release-readiness-team",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=60)

assert gate_decisions[0]["decision"] == "NO-GO"
assert gate_decisions[-1]["decision"] == "GO"
assert not review_findings["test-analyst"]["evidence"]["passed"]
assert review_findings["security-reviewer"]["evidence"]["findings"]
assert dossiers[0]["final_gate"]["decision"] == "GO"
assert {item["correlation_id"] for item in message_trail} == {"release-rc-140"}


## Inspect the system

Spans are stored only after finalization. The inspection checks end times,
durations, and both failure and success status so a zero-duration or perpetually
open trace cannot satisfy certification.


In [ ]:
trace_ids = trace_store.get_recent_traces(limit=100)
spans = [span for trace_id in trace_ids for span in trace_store.get_trace(trace_id)]
assert spans and all(span["end_time"] is not None for span in spans)
assert all(span["duration_ms"] > 0 for span in spans)
assert {span["status"] for span in spans} >= {"OK", "ERROR"}
assert any(event["tool"] == "apply_safe_patch" for event in tool_events)
assert sum(event["tool"] == "apply_safe_patch" for event in tool_events) == 1

show_message_graph(message_trail)
show_spore(next(spore for spore in received_spores if spore.knowledge["type"] == "initial_gate_decision"),
           "Actual initial no-go Spore")
show_json({"tool_events": tool_events, "spans": spans}, "Tool and trace evidence")
show_timeline()


In [ ]:
show_scorecard([
    ("Initial gate", gate_decisions[0]["decision"], "pass"),
    ("Real candidate test failed", review_findings["test-analyst"]["evidence"]["returncode"], "pass"),
    ("Security rule found", "SEC-001", "pass"),
    ("Bounded patches", 1, "pass"),
    ("Verification", "all checks pass", "pass"),
    ("Final gate", gate_decisions[-1]["decision"], "pass"),
    ("Finalized spans", len(spans), "pass"),
])
show_artifact("Release dossier", dossiers[0])


## Failure modes and tradeoffs

The static security check is intentionally narrow; it proves orchestration, not
the completeness of a production scanner. The safe patch is predefined rather
than model-authored, which makes the offline gate repeatable and limits writes
to reviewed files. A real system would require stronger sandboxing and signed
remediation artifacts.


## Extensions

Add dependency-license review, wheel installation, or a patch whose verification
still fails. Keep the final gate independent from the remediation planner.

## Cleanup

The temporary workspace, trace database, Agents, Reef, and tools are removed.
The committed fixture remains byte-for-byte unchanged.


In [ ]:
workspace_root = Path(workspace_context.name)
for function in agents:
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
reset_trace_store(); reset_tracer(); reset_config()
workspace_context.cleanup()
assert not workspace_root.exists()
show_callout("Release workspace removed", "Candidate copy and local trace store were deleted.", role="reef")
